# 读取csv

In [ ]:
import pandas as pd

# 1) 读取 CSV
file_path = "../data/5.清洗结果/clean_transport_result_1_4.csv"  # 改成你的文件路径
df = pd.read_csv(file_path)

# 2) 显示前 5 行
df.head(5)

# 将位置拼起来

In [ ]:
import pandas as pd

cols_to_join = ["province", "city", "district", "specific_place"]

for i, row in df.iterrows():
    parts = []
    for c in cols_to_join:
        v = row[c]

        # 先把真正的缺失值 NaN/None 变成空串
        if pd.isna(v):
            v = ""
        else:
            v = str(v).strip()
            # 2) 防止本来就是字符串 "nan"（或 "NaN"）这种情况
            if v.lower() == "nan":
                v = ""

        # 每一列都追加，保证输出长度固定为4
        parts.append(v)

    address = parts[0] + parts[1] + parts[2] + parts[3]
    print(parts)

# 百度地理编码：https://lbsyun.baidu.com/faq/api?title=webapi/guide/webservice-geocoding

In [ ]:
import os
# encoding:utf-8
import requests 

# 服务地址
host = "https://api.map.baidu.com"

# 接口地址
uri = "/geocoding/v3/"

# 此处填写你在控制台-应用管理-创建应用后获取的AK
ak = os.environ["BAIDU_MAP_AK"]  # 具体访问参数官网
params = {"city": "晋城市",
          "address": "山西省晋城市阳城县阳济公路涝泉村口路段",
          "ret_coordtype": "wgs84",
          "output": "json",
          "extension_analys_level": 1,
          # "extension_poi_infos": True,
          "ak": ak,}

# 请求解析地址
try:
    r = requests.get(host + uri, params=params, timeout=10)
    r.raise_for_status()  # HTTP 非 200 会抛异常
    data = r.json()

    # 百度 API：status == 0 表示成功
    if data.get("status") != 0:
        raise RuntimeError(f"API error: status={data.get('status')}, msg={data.get('msg')}")

    result = data.get("result") or {}
    location = result.get("location") or {}

    lng = location.get("lng")
    lat = location.get("lat")
    level = result.get("level")

    print(f"lng={lng}, lat={lat}, level={level}")

except Exception as e:
    print("请求或解析失败：", e)

# 合并代码

## to_csv保存

In [ ]:
# encoding:utf-8
import pandas as pd
import requests
import time
import os

# 读取 CSV
file_path = "../data/5.清洗结果/clean_transport_result_1_4.csv"  # 改成你的文件路径
# 保存csv路径
out_path = "../data/6.匹配地点/address_transport_result_1_4.csv"

# ✅ 断点续传 —— 如果已经有输出文件，就从输出文件继续
if os.path.exists(out_path):
    print(f"检测到已存在输出文件，继续断点续传：{out_path}")
    df = pd.read_csv(out_path)
else:
    df = pd.read_csv(file_path)

# ✅ 保证列存在（旧文件/新文件都兼容）
for col in ["lng", "lat", "level"]:
    if col not in df.columns:
        df[col] = ""
        
# 使用的列
cols_to_join = ["province", "city", "district", "specific_place"]

for i, row in df.iterrows():
    # ✅ 检测已处理过的就跳过（lng/lat 都有值）
    lng0 = df.at[i, "lng"]
    lat0 = df.at[i, "lat"]
    def _has_value(x):
        if pd.isna(x):
            return False
        s = str(x).strip()
        return s != "" and s.lower() != "nan"

    if _has_value(lng0) and _has_value(lat0):
        print(f"跳过{i}: 已有lng/lat")
        continue
    
    parts = []
    for c in cols_to_join:
        v = row[c]

        # 先把真正的缺失值 NaN/None 变成空串
        if pd.isna(v):
            v = ""
        else:
            v = str(v).strip()
            # 2) 防止本来就是字符串 "nan"（或 "NaN"）这种情况
            if v.lower() == "nan":
                v = ""

        # 每一列都追加，保证输出长度固定为4
        parts.append(v)

    address = parts[0] + parts[1] + parts[2] + parts[3]
    # print(address)
    
    
    # ------------------------------------------------------------------------------------------------   
    host = "https://api.map.baidu.com"   # 服务地址
    uri = "/geocoding/v3/"   # 接口地址
    ak = os.environ["BAIDU_MAP_AK"]  # 此处填写你在控制台-应用管理-创建应用后获取的AK
    # 具体访问参数官网
    params = {"city": parts[1],
              "address": address,
              "ret_coordtype": "wgs84",
              "output": "json",
              "extension_analys_level": 1,
              # "extension_poi_infos": True,
              "ak": ak,}

    
    lng = lat = level = ""   # ✅ 默认值，失败也能写回
    # 请求解析地址
    try:
        r = requests.get(host + uri, params=params, timeout=10)
        r.raise_for_status()  # HTTP 非 200 会抛异常
        data = r.json()
    
        # 百度 API：status == 0 表示成功
        if data.get("status") != 0:
            raise RuntimeError(f"API error: status={data.get('status')}, msg={data.get('msg')}")
    
        result = data.get("result") or {}
        location = result.get("location") or {}
    
        lng = location.get("lng")
        lat = location.get("lat")
        level = result.get("level")
    
        print(f"正在处理{i}:, lng={lng}, lat={lat}, level={level}")
        time.sleep(0.5)
    
    except Exception as e:
        print("请求或解析失败：", e)

    finally:
        # ✅ 处理一条写回一条：写入df新列 + 立刻保存csv
        df.at[i, "lng"] = lng
        df.at[i, "lat"] = lat
        df.at[i, "level"] = level
        print(len(df))
        df.to_csv(out_path, index=False, encoding="utf-8-sig")

## 流式输入输出

In [ ]:
# encoding:utf-8
import pandas as pd
import requests
import time
import os
import csv

file_path = "../data/5.清洗结果/clean_transport_result_1_4.csv"
out_path  = "../data/6.匹配地点/address_transport_result_1_4.csv"
tmp_path  = out_path + ".tmp"

os.makedirs(os.path.dirname(out_path), exist_ok=True)
os.makedirs(os.path.dirname(tmp_path), exist_ok=True)

cols_to_join = ["province", "city", "district", "specific_place"]

def _has_value(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    return s != "" and s.lower() != "nan"

# ✅ 1) 如果上次中断留下了 tmp，先恢复成正式 out_path（这样进度不会“只剩tmp”）
if os.path.exists(tmp_path) and os.path.getsize(tmp_path) > 0:
    # 如果 out_path 不存在，或 tmp 更新/更大，就用 tmp 覆盖 out_path
    if (not os.path.exists(out_path)) or (os.path.getmtime(tmp_path) >= os.path.getmtime(out_path)):
        print(f"检测到中断残留 tmp，恢复进度：{tmp_path} -> {out_path}")
        os.replace(tmp_path, out_path)

# ✅ 2) 统计已完成的行数（用于断点续传）
done_rows = 0
header = None
if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
    with open(out_path, "r", encoding="utf-8-sig", newline="") as f:
        r = csv.reader(f)
        header = next(r, None)
        done_rows = sum(1 for _ in r)
    print(f"断点续传：已完成 {done_rows} 行，继续处理剩余数据...")

# ✅ 3) 确定输出表头（保证有 lng/lat/level/ok）
if header is None:
    # 从原始文件读表头
    base_cols = list(pd.read_csv(file_path, nrows=0).columns)
    header = base_cols[:]
    for col in ["lng", "lat", "level", "ok"]:
        if col not in header:
            header.append(col)
else:
    # 旧输出缺列就补
    for col in ["lng", "lat", "level", "ok"]:
        if col not in header:
            header.append(col)

need_write_header = not (os.path.exists(out_path) and os.path.getsize(out_path) > 0)

# 百度配置（你原样保留）
host = "https://api.map.baidu.com"
uri = "/geocoding/v3/"
ak = os.environ["BAIDU_MAP_AK"]  # ✅ 4) 直接追加写 out_path：每行 writer.writerow + flush，中断也能保留
with open(out_path, "a", encoding="utf-8-sig", newline="") as f_out:
    writer = csv.DictWriter(f_out, fieldnames=header)
    if need_write_header:
        writer.writeheader()
        f_out.flush()

    # ✅ 5) 跳过已完成行（pandas 的 skiprows：0是表头不能跳）
    skiprows = range(1, done_rows + 1) if done_rows > 0 else None

    for chunk in pd.read_csv(file_path, skiprows=skiprows, chunksize=500):
        # 保证 chunk 里有这些列（否则 row.to_dict 可能缺）
        for col in ["lng", "lat", "level", "ok"]:
            if col not in chunk.columns:
                chunk[col] = ""

        for count, row in chunk.iterrows():
            row_dict = row.to_dict()

            # 组装 address（保持你原逻辑）
            parts = []
            for c in cols_to_join:
                v = row_dict.get(c, "")
                if pd.isna(v):
                    v = ""
                else:
                    v = str(v).strip()
                    if v.lower() == "nan":
                        v = ""
                parts.append(v)

            address = parts[0] + parts[1] + parts[2] + parts[3]

            params = {
                "city": parts[1],
                "address": address,
                "ret_coordtype": "wgs84",
                "output": "json",
                "extension_analys_level": 1,
                "ak": ak,
            }

            lng = lat = level = ""
            ok = 0  # 默认失败

            try:
                r = requests.get(host + uri, params=params, timeout=10)
                r.raise_for_status()
                data = r.json()

                if data.get("status") != 0:
                    raise RuntimeError(f"API error: status={data.get('status')}, msg={data.get('msg')}")

                result = data.get("result") or {}
                location = result.get("location") or {}

                lng = location.get("lng", "")
                lat = location.get("lat", "")
                level = result.get("level", "")
                ok = 1

                print(f"处理中: {count} address={address}, lng={lng}, lat={lat}, level={level}")


            except Exception as e:
                print("请求或解析失败：", e)

            # 写回当前行
            row_dict["lng"] = lng
            row_dict["lat"] = lat
            row_dict["level"] = level
            row_dict["ok"] = ok

            # ✅ 逐行写入 + 立刻 flush：中断也不会丢
            writer.writerow({k: row_dict.get(k, "") for k in header})
            f_out.flush()

            time.sleep(0.1)

# 检查ok=0识别的重新地理编码

In [ ]:
# encoding:utf-8
import csv
import os
import time
import requests

# 你保存的“结果文件”（里面有 ok/lng/lat/level）
out_path = "../data/6.匹配地点/address_transport_result_1w.csv"
tmp_path = out_path + ".retry.tmp"
bak_path = out_path + ".bak"  # 可选：备份

cols_to_join = ["province", "city", "district", "specific_place"]

# 百度配置（保持你原样）
host = "https://api.map.baidu.com"
uri = "/geocoding/v3/"
ak = os.environ["BAIDU_MAP_AK"]
session = requests.Session()


def norm(v):
    """把 None/nan/空白 统一处理成 ''"""
    if v is None:
        return ""
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return ""
    return s


def is_ok_zero(ok_val):
    """判断这一行是否需要重跑：ok=0 或 ok 缺失/非法 都算需要重跑"""
    s = norm(ok_val)
    if s == "":
        return True
    try:
        return int(float(s)) == 0
    except Exception:
        return True


def geocode(parts, max_retries=3, timeout=10):
    """调用百度地理编码：失败会重试，最后返回 (lng, lat, level, ok)"""
    address = "".join(parts)

    params = {
        "city": parts[1],  # city
        "address": address,
        "ret_coordtype": "wgs84",
        "output": "json",
        "extension_analys_level": 1,
        "ak": ak,
    }

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(host + uri, params=params, timeout=timeout)
            r.raise_for_status()
            data = r.json()

            if data.get("status") != 0:
                raise RuntimeError(f"API error: status={data.get('status')}, msg={data.get('msg')}")

            result = data.get("result") or {}
            location = result.get("location") or {}

            lng = location.get("lng", "")
            lat = location.get("lat", "")
            level = result.get("level", "")
            return lng, lat, level, 1

        except Exception as e:
            last_err = e
            # 简单退避
            time.sleep(1.0 * attempt)

    print(f"重试仍失败：address={address} err={last_err}")
    return "", "", "", 0


def main():
    if not os.path.exists(out_path) or os.path.getsize(out_path) == 0:
        raise FileNotFoundError(f"找不到结果文件或文件为空：{out_path}")

    # 如果上次重跑中断留下 tmp，可以先不管；我们这次会直接覆盖写 tmp_path
    if os.path.exists(tmp_path):
        os.remove(tmp_path)

    total = 0
    retry_total = 0
    retry_success = 0

    with open(out_path, "r", encoding="utf-8-sig", newline="") as f_in:
        reader = csv.DictReader(f_in)
        header = reader.fieldnames or []

        # 确保输出列完整
        for col in ["lng", "lat", "level", "ok"]:
            if col not in header:
                header.append(col)

        with open(tmp_path, "w", encoding="utf-8-sig", newline="") as f_out:
            writer = csv.DictWriter(f_out, fieldnames=header)
            writer.writeheader()
            f_out.flush()

            for row in reader:
                total += 1

                if is_ok_zero(row.get("ok")):
                    retry_total += 1
                    parts = [norm(row.get(c)) for c in cols_to_join]
                    lng, lat, level, ok = geocode(parts)

                    row["lng"] = lng
                    row["lat"] = lat
                    row["level"] = level
                    row["ok"] = ok

                    if ok == 1:
                        retry_success += 1

                    print(f"[{total}] 重跑 ok={ok} address={''.join(parts)} lng={lng} lat={lat} level={level}")
                    time.sleep(0.5)

                # 写入（保持原行顺序；ok=1 的行原样写回）
                writer.writerow({k: row.get(k, "") for k in header})
                f_out.flush()

    # 可选：备份原文件
    # 若你不想备份，注释掉下面两行即可
    # if os.path.exists(bak_path):
    #     os.remove(bak_path)
    os.replace(out_path, bak_path)

    # 用新文件覆盖原结果文件（原子替换）
    os.replace(tmp_path, out_path)

    print(f"完成：总行数={total}；需要重跑={retry_total}；重跑成功={retry_success}")
    print(f"已更新：{out_path}")
    print(f"备份文件：{bak_path}")


if __name__ == "__main__":
    main()
